# Institutional Quant Research Engine V2.1

Thin reproducible interface. **All core logic lives in `src/quant_research/`**
- this notebook only orchestrates: configuration -> data -> PIT validation ->
features -> walk-forward baseline -> robustness -> ablation -> experiment
record -> promotion status.

Research contract: never tune on test; validation is the selection layer;
test is evaluation-only; synthetic results are NOT market evidence.

## 1. Configuration

In [3]:
from quant_research.config import load_config
from quant_research import __version__

CONFIG_PATH = "configs/baseline.yaml"   # switch to configs/real_spy.yaml for real data
cfg = load_config(CONFIG_PATH)
print(f"engine {__version__} | config fingerprint: {cfg.fingerprint()}")
cfg.to_dict()

engine 2.1.3 | config fingerprint: ac67953774155e0f


## 2. Data load & validation

In [5]:
from quant_research.data.loaders import load_market_data, to_panels
from quant_research.data.validation import validate_ohlcv, missing_data_report
from quant_research.data.snapshots import save_snapshot

ohlcv, data_meta = load_market_data(cfg.data)
ohlcv = validate_ohlcv(ohlcv)          # fails loudly on schema violations
snapshot = save_snapshot(ohlcv, cfg.data.raw_snapshot_dir, name=f"{cfg.data.mode}_ohlcv")
missing = missing_data_report(ohlcv)
print(f"dataset_hash={snapshot['dataset_hash']} rows={len(ohlcv)}")
missing

dataset_hash=5da40b7b90035c78 rows=24640


## 3. Point-in-time event validation

In [7]:
from quant_research.run import generate_synthetic_events
from quant_research.features.point_in_time import validate_events

# Synthetic mode exercises the PIT layer with clearly-labelled synthetic
# events; real information feeds enter through the same validated schema.
events = None
if cfg.data.mode == "synthetic":
    events = validate_events(generate_synthetic_events(
        ohlcv[ohlcv.symbol == cfg.data.target]["timestamp"], cfg.data.target))
    print("PIT event schema validated (synthetic events; NOT market evidence)")

PIT event schema validated (synthetic events; NOT market evidence)


## 4. Feature construction + leakage check

In [9]:
from quant_research.features.price_volume import build_price_volume_features
from quant_research.features.information import build_information_features
from quant_research.features.leakage import feature_leakage_report
from quant_research.features.registry import registry_hash

close, volume = to_panels(ohlcv)
price_feats = build_price_volume_features(close, volume, cfg.data.target)
if events is not None:
    info_feats = build_information_features(close.index, events, cfg.data.target)
    features = price_feats.join(info_feats, how="left")
else:
    features = price_feats
leakage = feature_leakage_report(close, volume, cfg.data.target)
print(f"features={features.shape} feature_version={registry_hash(list(features.columns))}")
print("leakage check passed:", leakage["passed"])

features=(3520, 19) feature_version=9304939ac943d318
leakage check passed: True


## 5. Walk-forward baseline (TRAIN -> VAL -> PURGE/EMBARGO -> TEST)

In [11]:
import numpy as np
from quant_research.evaluation.walk_forward import LockedTestProtocol
from quant_research.experiments.registry import TrialCounter
from quant_research.strategies.baseline import run_walk_forward, summarize_experiment

y = (close[cfg.data.target].shift(-1) > close[cfg.data.target]).astype("float")
y[close[cfg.data.target].shift(-1).isna()] = np.nan
fwd = close[cfg.data.target].shift(-1) / close[cfg.data.target] - 1.0

locked_test = LockedTestProtocol()          # freeze the test layout
counter = TrialCounter("artifacts/trial_counter.json")
baseline = run_walk_forward(features, y, fwd, cfg,
                            locked_test=locked_test, trial_counter=counter)
summary = summarize_experiment(baseline)
print(f"global trials: {counter.count}")
baseline.folds

global trials: 63


## 6. Robustness battery

In [13]:
from quant_research.evaluation.robustness import (
    cost_stress, slippage_stress, delay_stress)

# Stresses run on the EXACT fold-level OOS execution path: same folds,
# selected features, parameters, per-fold thresholds, preprocessing,
# sizing.  Only the stressed variable changes.
cost_table = cost_stress(features, y, fwd, cfg, baseline, locked_test)
slip_table = slippage_stress(features, y, fwd, cfg, baseline, locked_test)
delay_table = delay_stress(features, y, fwd, cfg, baseline, locked_test)
display(cost_table); display(slip_table); display(delay_table)


     sharpe  gross_sharpe  net_return  ...  fee_bps  slippage_bps  delay_bars
0  0.226690      0.250197    0.068950  ...      0.0           1.0           0
1  0.167901      0.250197    0.048540  ...      2.5           1.0           0
2  0.109108      0.250197    0.028515  ...      5.0           1.0           0
3 -0.008364      0.250197   -0.010402  ...     10.0           1.0           0
4 -0.241962      0.250197   -0.083907  ...     20.0           1.0           0

[5 rows x 12 columns]
     sharpe  gross_sharpe  net_return  ...  fee_bps  slippage_bps  delay_bars
0  0.132623      0.250197    0.036479  ...      5.0           0.0           0
1  0.109108      0.250197    0.028515  ...      5.0           1.0           0
2  0.085596      0.250197    0.020612  ...      5.0           2.0           0
3  0.015110      0.250197   -0.002737  ...      5.0           5.0           0
4 -0.102090      0.250197   -0.040481  ...      5.0          10.0           0

[5 rows x 12 columns]
     sharpe  gross

## 7. Information-source ablation + empirical null

In [15]:
import pandas as pd
from quant_research.evaluation.placebo import run_placebo_null, placebo_statistics

def _summarize(feats):
    return summarize_experiment(run_walk_forward(feats, y, fwd, cfg, locked_test=locked_test))

ablation = [{"source": "price_volume",
             "mean_oos_sharpe": _summarize(price_feats)["mean_oos_sharpe"]}]
if events is not None:
    ablation.append({"source": "price_plus_information",
                     "mean_oos_sharpe": summary["mean_oos_sharpe"]})
null = run_placebo_null(features, y, fwd, lambda X, yy, ff: _summarize(X),
                        n_runs=cfg.research.placebo_runs, seed=cfg.model.random_seed)
placebo = placebo_statistics(summary["mean_oos_sharpe"], null)
display(pd.DataFrame(ablation))
print("placebo:", placebo)

                   source  mean_oos_sharpe
0            price_volume        -0.683038
1  price_plus_information         0.119550
placebo: {'percentile': 0.7, 'adjusted_p': 0.3333333333333333, 'percentile_mc_se': 0.10246950765959599, 'null_mean': -0.10827904348552844, 'null_median': -0.10427838433562667, 'null_std': 0.41757944747818593, 'null_p95': 0.4383760359074652, 'n_runs': 20, 'observed': 0.11955027808235123}


## 8. Experiment record + promotion status

In [17]:
from quant_research.run import run_research_pipeline

# Full pipeline execution (same config): registers the immutable experiment
# record, writes artifacts, and applies the promotion gates.
report = run_research_pipeline(cfg, cfg.output_dir)
record = report["experiment_record"]
print("experiment_id:", record["experiment_id"])
print("evidence_status:", record["evidence_status"])
print("promotion_state:", record["promotion_state"])
print("failed gates:", record["failed_gates"] or "none")
report["leaderboard"]

experiment_id: 20260910T115619Z_1b68f811212545b0
evidence_status: SYNTHETIC_OFFLINE
promotion_state: RESEARCH_ONLY
failed gates: ['cost_stress_survives', 'placebo_separates']
